[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Reading Shape Errors


## What you will be able to do

Read a shape error from PyTorch, Keras or TensorFlow and say which tensor had the wrong size, along
which dimension, and what the layer expected, and find the layer at fault by printing the shape
after every layer.

## The idea

### The problem

A network for the made-up stations takes four features. Give it a table with a fifth column, or a
single station with no batch dimension, or a hidden layer whose size was counted wrong, and it
raises an error that names two shapes and four or five numbers. The message is exact, but it
describes the arithmetic inside a layer, a step away from the line you wrote. PyTorch talks about
`mat1` and `mat2`, TensorFlow about `In[0]` and `In[1]`, and Keras about `axis -1`. Reading one
means knowing what each framework calls each tensor, and which of its numbers is the batch.

The fix is usually one number in one place. Finding that place is quick once you know what
each number in the message refers to.

### A formal definition

> A tensor's **shape** lists its size along each of its dimensions. In both frameworks the first
> dimension of a linear layer's input is the **batch dimension**, one entry for each example, and
> the layer reads the **last** dimension as the features. A **shape error** is raised when an operation
> receives a tensor whose size along some dimension is not the size the operation needs.

### Why it works that way

`nn.Linear(4, 32)` computes `x @ W.T + b`, and its weight `W` is (32, 4), so `W.T` is (4, 32). A
matrix product of (8, 4) and (4, 32) gives (8, 32). The two inner numbers must be equal, and the
outer two give the output. When they differ, PyTorch reports both shapes, the input first as `mat1`
and the transposed weight second as `mat2`. The batch size is the first number of `mat1`, and it
never causes the error.

Keras checks an input against what each layer expects before it computes anything, so its message
names the layer and the axis that is wrong, where `-1` means the last one. `fit` and `predict` run
their step as a TensorFlow graph, traced for the shapes of the first data they are given. When later
data has another shape, TensorFlow traces the step again and leaves the size that changed unknown,
so Keras's check has nothing to compare. TensorFlow's own matrix product then reports the mismatch
as the graph runs, as `In[0]` and `In[1]`, the input and the layer's kernel, under the heading
"Graph execution error".

The two frameworks also differ on a single example. PyTorch's `nn.Linear` accepts a tensor with no
batch dimension and treats it as one example, while Keras raises an error without it, and writes
its expected shape with `None` in the batch's place.

### Where this shows up

The **Convolutional Networks** notebook meets channels in the wrong position and a flattened size
counted wrong, the **Recurrent Networks** notebook a time axis in the wrong place, and the **Losses
and Optimizers** notebook a target whose shape differs from the prediction's, which can broadcast
without any error.

### What this notebook covers

- what a linear layer multiplies, and the shapes on each side
- PyTorch's message for one column too many, number by number
- Keras's message for the same mistake, and what axis -1 means
- where each framework says what it expects
- the shape after every layer, printed in both frameworks
- three errors, from a miscounted hidden layer, from Keras's models and from inside `fit`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch
from torch import nn

layer = nn.Linear(4, 1)
layer(torch.ones(32, 5))
```

```
RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x5 and 4x1)
```

That is the last line of the error. `32x5` is the input, 32 examples with 5 features each, and `4x1`
is the layer's weight turned around, 4 features in and 1 out. The 5 and the 4 must be equal.

## Setup

Two libraries and their layers, a way to quiet TensorFlow while it loads, and the stations.

- `numpy` builds the stations, and adds the column too many that the errors come from
- `torch` is PyTorch, and `nn` its module of layers
- `tensorflow` is TensorFlow, and `keras` its library of layers and models
- `re` takes the codes for bold text out of Keras's messages before a cell prints them
- `contextlib` and `os` hide the messages TensorFlow prints while it loads

TensorFlow's C++ code prints several lines as it loads, each stamped with the time and a process
number, so no two runs would show the same output. Setting `TF_CPP_MIN_LOG_LEVEL` to 3 hides what
it logs once it has started, and `c_messages_hidden` sends the lines it prints while it starts to
`os.devnull`. The cell prints PyTorch's version without the part after `+`, which names the build.

`station_network_arrays` makes the 1,200 weather stations in Norway that the **scikit-learn, Deep
Dive** guide uses. The stations are made up, and a generator started from a fixed seed draws them,
so they are the same on every run. `scaled` holds each station's latitude, longitude, elevation and
distance from the coast, standardized with the training stations' means and standard deviations,
`jan_mean` its January mean and `snow_on_may_1` a 1 for snow on 1 May. The cell splits the stations
into 800 for training, 200 for validation and 200 for testing.

In [1]:
import contextlib
import os
import re

import numpy as np
import torch
from torch import nn


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


## Worked examples

### What a linear layer multiplies

A network for the stations starts with `nn.Linear(4, 32)`. Eight stations go in, and the cell prints
the shape of the input, of the weight, of the weight transposed and of the output.

In [2]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 1))

x = torch.from_numpy(scaled[train_rows[:8]])
first = net[0]
print("input         ", tuple(x.shape))
print("weight        ", tuple(first.weight.shape))
print("weight.T      ", tuple(first.weight.T.shape))
print("first output  ", tuple(first(x).shape))
print("model output  ", tuple(net(x).shape))

input          (8, 4)
weight         (32, 4)
weight.T       (4, 32)
first output   (8, 32)
model output   (8, 1)


Eight stations of four features go in, and the weight, turned around, is 4 by 32. The inner numbers,
4 and 4, match, and the outer ones, 8 and 32, give the first layer's output. The last layer gives one
prediction for each of the eight stations.

### PyTorch's message for an extra column

`five_columns` is the four standardized features with `snow_on_may_1` added as a fifth column, the
kind of table that results when a column is added to the data and not to the model. `shape_error`
runs an action and prints the kind of error it raises with the line of its message that gives the
sizes, so that one cell can show several.

In [3]:
five_columns = np.column_stack([scaled, snow_on_may_1]).astype("float32")


def shape_error(action):
    """Run action, and print the error it raises with the line of its message that gives the sizes."""
    try:
        action()
    except Exception as error:
        text = re.sub(r"\x1b\[[0-9;]*m", "", str(error))
        line = next(line for line in text.splitlines() if "shape" in line or "size" in line)
        print(f"{type(error).__name__}: {line}")
    else:
        print("no error")


shape_error(lambda: net(torch.from_numpy(five_columns[train_rows[:32]])))
shape_error(lambda: net(torch.from_numpy(five_columns[train_rows[:7]])))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x5 and 4x32)
RuntimeError: mat1 and mat2 shapes cannot be multiplied (7x5 and 4x32)


Read the first message from the left: 32 stations of 5 columns each, then the first layer's weight
turned around, 4 features in and 32 out. The 5 and the 4 are the mismatch. In the second message
only the first number changed, to 7, because the batch size never causes this error.

### Keras's message for the same mistake

The cell builds the same network in Keras, with its layers named, and gives it the same 32
stations of five columns.
Calling the `Dense` layer named `hidden` on its own shows the check Keras makes.

In [4]:
keras.utils.set_random_seed(0)
net_keras = keras.Sequential([
    keras.Input((4,), name="features"),
    keras.layers.Dense(32, activation="relu", name="hidden"),
    keras.layers.Dense(1, name="output"),
], name="station_net")

shape_error(lambda: net_keras.get_layer("hidden")(five_columns[train_rows[:32]]))

ValueError: Input 0 with name 'None' of layer 'hidden' is incompatible with the layer: expected axis -1 of input shape to have value 4, but received input with shape (32, 5)


Keras names the layer, `hidden`, and the axis, `-1`, which is the last one, where the features are.
It expected 4 there and received an input of shape (32, 5). `Input 0` is the layer's first and only
input.

### Where each framework says what it expects

Before reading a message, it helps to know the shape a model wants. A PyTorch linear layer keeps its
sizes as `in_features` and `out_features`. A Keras model built from a `keras.Input` knows the shape
of its input and output, with `None` for the batch.

In [5]:
print("PyTorch:", net[0].in_features, "features in,", net[-1].out_features, "out")
print("Keras:  ", net_keras.input_shape, "in,", net_keras.output_shape, "out")
print("data:   ", scaled.shape, "and", five_columns.shape)

PyTorch: 4 features in, 1 out
Keras:   (None, 4) in, (None, 1) out
data:    (1200, 4) and (1200, 5)


Both models take four features in and give one value out, and `five_columns` has five. Keras writes
the batch dimension as `None`, because it accepts any number of stations.

### The shape after every layer

When a model is longer than one layer, a direct way to find a mismatch is to pass a small batch
through it one layer at a time and print the shape each layer gives. An `nn.Sequential` can be
looped over, and so can a Keras model's `layers`.

In [6]:
x = torch.from_numpy(scaled[train_rows[:8]])
for layer in net:
    x = layer(x)
    print(f"{type(layer).__name__:<7} {tuple(x.shape)}")

Linear  (8, 32)
ReLU    (8, 32)
Linear  (8, 1)


In [7]:
x = scaled[train_rows[:8]]
for layer in net_keras.layers:
    x = layer(x)
    print(f"{layer.name:<7} {tuple(x.shape)}")

hidden  (8, 32)
output  (8, 1)


Each line is one layer's output. In PyTorch the `ReLU` keeps the shape it was given, and in Keras it
sits inside the `hidden` layer, so the Keras loop prints one line fewer.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/07-reading-shape-errors-solutions.ipynb).

**1.** Use `shape_error` to give `net` the first 10 test stations of `five_columns`, and write down what each of the four numbers in the message is.

In [8]:
# your code here

**2.** This model raises a shape error: `nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(4, 1))`. Print the shape after every layer for 8 stations to find the layer at fault, then fix it and print the output's shape.

In [9]:
# your code here

**3.** Build a Keras model for `five_columns`, with `keras.Input((5,))`, a hidden `Dense` layer of 16 and one output, and print the shape of its predictions for the first three test stations.

In [10]:
# your code here

**4.** Predict the January mean for the first test station alone, with `net` and with `net_keras`, giving each the station as a batch of one with `reshape(1, -1)`, and print the shape of each answer.

In [11]:
# your code here

**5.** Give `nn.Linear(4, 1)` a tensor of shape (2, 3, 4) through `shape_error`, then print the shape of what it returns. Which dimensions does the layer treat as the batch?

In [12]:
# your code here

## Common errors

### `RuntimeError: mat1 and mat2 shapes cannot be multiplied (8x16 and 8x1)`

This network's second linear layer was written with 8 inputs, where the layer before it gives 16.

In [13]:
torch.manual_seed(0)
miscounted = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(8, 1))
miscounted(torch.from_numpy(scaled[train_rows[:8]]))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (8x16 and 8x1)

Read from the left: 8 stations, each with 16 values, which is the output of the first linear layer,
and a weight turned around to 8 by 1, from a layer that takes 8 inputs. The data has the four
features the first layer takes, so the mismatch is inside the model, between its two linear layers.

In [14]:
x = torch.from_numpy(scaled[train_rows[:8]])
for layer in miscounted:
    print(f"{type(layer).__name__:<7} in {tuple(x.shape)}", end="")
    if isinstance(layer, nn.Linear):
        print(f" | expects {layer.in_features} features")
    else:
        print()
    if isinstance(layer, nn.Linear) and x.shape[-1] != layer.in_features:
        break
    x = layer(x)

Linear  in (8, 4) | expects 4 features
ReLU    in (8, 16)
Linear  in (8, 16) | expects 8 features


The loop prints what each layer is given, and what each linear layer expects, and stops at the first
that disagrees. The second linear layer is given 16 values and expects 8, so change it to
`nn.Linear(16, 1)`.

### `ValueError: Exception encountered when calling Sequential.call().`

A Keras model built in this cell is asked for predictions on the 200 test stations of the
five-column table. It is new because an earlier `predict` would change where the error comes from,
as the last error in this section explains.

In [15]:
keras.utils.set_random_seed(0)
fresh_keras = keras.Sequential([
    keras.Input((4,), name="features"),
    keras.layers.Dense(32, activation="relu", name="hidden"),
    keras.layers.Dense(1, name="output"),
], name="station_net")
fresh_keras.predict(five_columns[test_rows], verbose=0)

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'hidden' is incompatible with the layer: expected axis -1 of input shape to have value 4, but received input with shape (32, 5)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32, 5), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>

The first line names only the model. The bold line under it names the layer, and says that
`hidden` expected 4 values along axis -1 and received an input of shape (32, 5). The 32 is not the
200 test stations. `predict` works through its input in batches of 32, and the first batch is the
one that failed. Under that, `Arguments received` lists what the model was given. Give the model the
four features it was built for, or build it for five with `keras.Input((5,))`.

The same first line comes from a single station given with no batch dimension, which is how a
prediction for one place is usually first written.

In [16]:
one_station = scaled[test_rows[0]]
print(one_station.shape)
fresh_keras(one_station)

(4,)


ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input [ 0.5471716   0.68878543 -0.48540717 -0.6121734 ] with name 'features' and path ''. Expected shape (None, 4), but input has incompatible shape (4,)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(4,), dtype=float32)
  • training=None
  • mask=None
  • kwargs=<class 'inspect._empty'>

Here the bold line says the model expected `(None, 4)`, any number of stations with four features
each, and received `(4,)`, four numbers with no batch dimension. It prints the input itself too.

In [17]:
with torch.no_grad():
    print("PyTorch, no batch dimension:", tuple(net(torch.from_numpy(one_station)).shape))
    print("PyTorch, a batch of one:    ", tuple(net(torch.from_numpy(one_station.reshape(1, -1))).shape))
print("Keras, a batch of one:      ", net_keras.predict(one_station.reshape(1, -1), verbose=0).shape)

PyTorch, no batch dimension: (1,)
PyTorch, a batch of one:     (1, 1)
Keras, a batch of one:       (1, 1)


PyTorch's `nn.Linear` accepted the station with no batch dimension and returned one number in a
shape of `(1,)`, where a batch of one gives `(1, 1)`, as Keras does. `reshape(1, -1)` makes one row
with as many columns as the array has, which works in both.

### `InvalidArgumentError: Graph execution error:`

`fit` trains on the four features for one epoch, and then a second `fit` is given the five-column
table. The message of the error from the second call names nodes of the compiled graph, and ends with
a number that counts the functions TensorFlow has compiled earlier in the session, so it changes
when cells are run again. The cell catches the error and prints its first line and the line with
the sizes.

In [18]:
net_keras.compile(optimizer="adam", loss="mse")
net_keras.fit(scaled[train_rows], jan_mean[train_rows], epochs=1, verbose=0)

try:
    net_keras.fit(five_columns[train_rows], jan_mean[train_rows], epochs=1, verbose=0)
except tf.errors.InvalidArgumentError as error:
    lines = str(error).splitlines()
    print(f"{type(error).__name__}: {lines[0]}")
    print([line for line in lines if "Matrix size-incompatible" in line][0])

InvalidArgumentError: Graph execution error:
Matrix size-incompatible: In[0]: [32,5], In[1]: [4,32]


The first `fit` traced the training step for batches of 32 stations with 4 features. The second
`fit` gave it batches with 5 columns, so TensorFlow traced the step again with the number of columns
left unknown, and Keras's check had no number to compare with 4. The error came from TensorFlow's
matrix product as the graph ran, which describes the mismatch in its own terms. `In[0]` is the
input, a batch of 32 stations with 5 columns, and `In[1]` is the hidden layer's kernel, 4 by 32,
inputs by outputs, so the numbers that must match are the 5 and the 4. A Keras kernel is already
inputs by outputs, so it appears in the message as it is stored.

## Recap

- You can read PyTorch's `mat1` and `mat2`, Keras's `axis -1` and TensorFlow's `In[0]` and `In[1]`,
  and say which number is the batch, which the features given and which the features expected.
- You can find what a model expects, from `in_features` in PyTorch and `input_shape` in Keras.
- You can find the layer at fault by passing a small batch through a model one layer at a time.
- You can give either framework a single example as a batch of one.

## What is next

The **Losses and Optimizers** notebook chooses the loss for a regression and for a classification
and the optimizer that makes the step, and shows a target whose shape broadcasts against the
prediction without an error.

---

&#8592; **Previous:** [Layers and Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/06-layers-and-models.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Losses and Optimizers](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/08-losses-and-optimizers.ipynb) &#8594;
